In [ ]:
pip install -U sentence-transformers

In [ ]:
pip install transformers

In [ ]:
pip install datasets

In [4]:
pip install tqdm

In [5]:
import torch
import os
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer
from transformers import pipeline
from tqdm.notebook import tqdm
from datasets import load_dataset,concatenate_datasets


DATASET_NAME ='ag_news'
MAX_LENGTH = 120


def prepare_dataset(name):
  train_ds = load_dataset(name, split='train[:66%]')
  val_ds = load_dataset(name, split='train[66%:]')
# train_ds = load_dataset(name, split='train')
  test_ds = load_dataset(name, split='test')

# train_ds = train_ds.map(lambda examples: {'labels': examples['label']})
# val_ds = val_ds.map(lambda examples: {'labels': examples['label']})
# test_ds = test_ds.map(lambda examples: {'labels': examples['label']})

# train_ds = train_ds.train_test_split(100)  # for experiment we only use 500 examples
# train_ds = train_ds['test']

# val_ds = val_ds.train_test_split(100)  # for experiment we only use 500 examples
# val_ds = val_ds['test']

# test_ds = test_ds.train_test_split(100)  # for experiment we only use 500 examples
# test_ds = test_ds['test']

  return train_ds,val_ds,test_ds

In [6]:
embedder = SentenceTransformer('roberta-large')

train_ds, val_ds, test_ds = prepare_dataset(DATASET_NAME) # train_ds used as pool of examples, test_ds as the target dataset
class_names = test_ds.features["label"].names

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

if DATASET_NAME == 'imdb': 
  unmasker = pipeline('fill-mask', model='bert-base-cased', targets=['good', 'bad'])
elif DATASET_NAME == 'ag_news':
  unmasker = pipeline('fill-mask', model='bert-base-cased', targets=['World','Business','Sports','Science','Technology'])
  tokenizer.add_tokens('Sci/Tech')
else:
  raise Exception('Unsupported Dataset')

# prepocess the dataset
train_ds = train_ds.map(lambda e: {'text' : tokenizer.decode(tokenizer.encode(e['text'], padding = True, truncation= True, max_length = MAX_LENGTH), skip_special_tokens= True)})
val_ds = val_ds.map(lambda e: {'text' : tokenizer.decode(tokenizer.encode(e['text'], padding = True, truncation= True, max_length = MAX_LENGTH), skip_special_tokens= True)})
test_ds = test_ds.map(lambda e: {'text' : tokenizer.decode(tokenizer.encode(e['text'], padding = True, truncation= True, max_length = MAX_LENGTH), skip_special_tokens= True)})


Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


  0%|          | 0/79200 [00:00<?, ?ex/s]

  0%|          | 0/40800 [00:00<?, ?ex/s]

  0%|          | 0/7600 [00:00<?, ?ex/s]

In [7]:
print(len(train_ds))
print(len(val_ds))
print(len(test_ds))

79200
40800
7600


In [8]:
# preprocess examples in train_ds as form of e = (sentence, label, embedding)
unlabeled_examples = [{'text': e['text'], 'label': e['label'],
              'embedding': embedder.encode(e['text'], convert_to_tensor=False)} for e in tqdm(train_ds)]
unlabeled_corpus_embeddings = torch.FloatTensor([example['embedding'] for example in unlabeled_examples])

labeled_examples = [{'text': e['text'], 'label': e['label'],
              'embedding': embedder.encode(e['text'], convert_to_tensor=False)} for e in tqdm(val_ds)]
labeled_corpus_embeddings = torch.FloatTensor([example['embedding'] for example in labeled_examples])


labeled_top_k = min(1, len(labeled_examples))

unlabeled_top_k = min(2, len(unlabeled_examples))

  0%|          | 0/79200 [00:00<?, ?it/s]

  0%|          | 0/40800 [00:00<?, ?it/s]

In [9]:
def label2name(label):
    if DATASET_NAME == 'imdb':
      if label == 1:
          return 'good'
      else:
          return 'bad'
    elif DATASET_NAME == 'ag_news':
      return class_names[label]
    else: 
      return 'null'


def generate_prompt(indices, query, Labeled=True):
    prompt = ''

    if Labeled:
      for id in indices:
        name = label2name(labeled_examples[id]['label'])
        if DATASET_NAME == 'imdb':
          prompt += labeled_examples[id]['text'] + ' The movie is ' + name + '.\n'
        elif DATASET_NAME == 'ag_news':
          if name == 'Sci/Tech':
            name = 'Science and Technology'
          prompt += labeled_examples[id]['text'] + ' The article is about ' + name + '.\n'
        else:
          raise Exception('Unsupported Dataset')
    else: # unlabeled priming
      for id in indices:
        if DATASET_NAME == 'imdb':
          tmp = unlabeled_examples[id]['text'] + ' The movie is [MASK].'
          prompt += unmasker(tmp)[0]['sequence'] +'\n '
        elif DATASET_NAME == 'ag_news':
          tmp = unlabeled_examples[id]['text'] + ' The article is about [MASK].'
          predicted = unmasker(tmp)[0]['token_str']
          if predicted == 'Science' or predicted == 'Technology':
            predicted = 'Science and Technology'
          prompt += unlabeled_examples[id]['text'] + ' The article is about ' + predicted + '.\n'
        else:
          raise Exception('Unsupported Dataset')

    return prompt

In [10]:

def predict(prompt):
    scores = unmasker(prompt)
    name = scores[0]['token_str']

    return name

In [11]:
# iterate over the whole test_dataset
correct_num = 0
total_num = 0
pbar = tqdm(test_ds, desc='Running')
labeled_corpus_embeddings = labeled_corpus_embeddings.to(device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
unlabeled_corpus_embeddings = unlabeled_corpus_embeddings.to(device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
for query in pbar:
    query_embedding = embedder.encode(query['text'], convert_to_tensor=True)

    # the unlabeled part
    unlabeled_cos_scores = util.pytorch_cos_sim(query_embedding, unlabeled_corpus_embeddings)[0]
    unlabeled_top_results = torch.topk(unlabeled_cos_scores, k=unlabeled_top_k)
    unlabeled_indices = unlabeled_top_results[1].tolist()
    unlabeled_indices.reverse()  # we put the more similar example, nearer to our query
    prompt = generate_prompt(unlabeled_indices, query['text'], Labeled=False)


    # the labeled part 
    labeled_cos_scores = util.pytorch_cos_sim(query_embedding, labeled_corpus_embeddings)[0]
    labeled_top_results = torch.topk(labeled_cos_scores, k=labeled_top_k)
    labeled_indices = labeled_top_results[1].tolist()
    labeled_indices.reverse()  # we put the more similar example, nearer to our query
    prompt += generate_prompt(labeled_indices, query['text'], Labeled=True)

    if DATASET_NAME == 'imdb':
      prompt += query['text'] + ' The movie is [MASK].'
    elif DATASET_NAME == 'ag_news':
      prompt += query['text'] + ' The article is about [MASK].'
    else:
      raise Exception('Unsupported Dataset')
    

    if DATASET_NAME == 'imdb':
      true_name = 'good' if query['label'] ==1 else 'bad'
    elif  DATASET_NAME == 'ag_news':
      true_name = class_names[query['label']]
    else:
      raise Exception('Unsupported Dataset')

    predicted = predict(prompt)

    if true_name == 'Sci/Tech' and (predicted=='Science' or predicted=='Technology'):
      correct_num += 1
    elif predicted == true_name:
      correct_num += 1
    total_num += 1
    pbar.set_postfix({' binary_accuracy': correct_num/total_num})

print('%d/%d  accuracy: %.4f'%(correct_num, total_num, correct_num/total_num))

Running:   0%|          | 0/7600 [00:00<?, ?it/s]

5684/7600  accuracy: 0.7479
